# Samples from models on 60km -> 2.2km-4x over Birmingham

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.cordex_ml_default_params import *

In [ ]:
import IPython
import matplotlib
import matplotlib.pyplot as plt

from mlde_analysis.data import prep_eval_data
from mlde_analysis.examples import plot_examples, em_timestamps
from mlde_analysis import platecarree

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

In [ ]:
examples_to_plot = {}

for source, percentiles in example_percentiles.items():
    em_ts = em_timestamps(EVAL_DS[source].assign_coords(ensemble_member=("ensemble_member", ["01"])), percentiles=percentiles, overrides=example_overrides[source])

    examples_to_plot[source] = em_ts

In [ ]:
for source, examples in examples_to_plot.items():
    IPython.display.display_html(f"<h2>{source} Samples</h2>", raw=True)

    if source == "CPM":
        fig_width = 6
    else:
        fig_width = 4
    fig_height = 4.5
    fig = plt.figure(layout="constrained", figsize=(fig_width, fig_height))
    plot_examples(
        EVAL_DS[source], examples,
        vars=eval_vars, models=MODELS[source], fig=fig, sim_title=source, examples_sample_idxs=examples_sample_idxs, inputs=example_inputs,
    )
    plt.show()

In [ ]:
import numpy as np

for var in eval_vars:
    IPython.display.display_markdown(f"## {var}", raw=True)
    
    for source, ds in EVAL_DS.items():
        IPython.display.display_markdown(f"### {source} Samples", raw=True)
        t_idxs = np.random.choice(np.flatnonzero(ds["time"]), min(15, len(EVAL_DS[source]["time"])), replace=False)
    
        IPython.display.display_markdown(f"#### Simulation", raw=True)
        g = ds[f"target_{var}"].isel(time=t_idxs, ensemble_member=0).plot(col="time", col_wrap=5)
        plt.show()
        
        for model, model_da in ds[f"pred_{var}"].isel(time=t_idxs, ensemble_member=0, sample_id=0).groupby("model"):
            IPython.display.display_markdown(f"#### {model}", raw=True)
            g = model_da.plot(col="time", col_wrap=5)
            plt.show()